# Continuing Training with a 7B Model
**Note:** To resume from a shared folder, go to 'Shared with me' in your new Google Drive, right-click the folder, and select **Organize > Add shortcut > My Drive** so Colab can see it naturally.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# DRIVE_PATH should point to your shortcut for the shared folder
DRIVE_PATH = "/content/drive/MyDrive/indicators_grpo_v2"
print(f'Looking for checkpoints in: {DRIVE_PATH}')

## Step 1: Upload and Extract Hackathon Zip

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload() # Select hackathon.zip from your Mac

with zipfile.ZipFile('hackathon.zip', 'r') as z:
    z.extractall('/content/')

%cd /content/hackathon
print(f'Ready in: {os.getcwd()}')

## Step 2: Install Dependencies

In [ ]:
!pip install numpy pandas trl peft yfinance accelerate bitsandbytes

## Step 3: Initialize Hugging Face Token

In [ ]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN_HERE")
print('Hugging Face Logged In!')

## Step 5: Check Progress & Find Latest Checkpoint

In [ ]:
import os

def get_latest_checkpoint(path):
    if not os.path.exists(path): return None
    checkpoints = [d for d in os.listdir(path) if d.startswith("checkpoint-")]
    if not checkpoints: return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    return os.path.join(path, latest)

LATEST = get_latest_checkpoint(DRIVE_PATH)
if LATEST:
  print(f"✅ FOUND CHECKPOINT: Resuming from {LATEST}")
  RESUME_ARG = f"--resume_from_checkpoint {LATEST}"
else:
  print(f"🆕 No checkpoint found in {DRIVE_PATH}. Check your shortcut name!")
  RESUME_ARG = ""


### 🔧 REPAIR RUN: Stabilizing weights from Step 160
The analysis shows the model decayed after 180 due to reward imbalance. We are resuming from the peak (Step 160) with the new symmetric reward function.

In [ ]:
# Overwrite LATEST to force resume from the peak stable version
REPAIR_CKPT = os.path.join(DRIVE_PATH, "checkpoint-160")
if os.path.exists(REPAIR_CKPT):
  LATEST = REPAIR_CKPT
  RESUME_ARG = f"--resume_from_checkpoint {LATEST}"
  print(f"✅ REPAIR MODE: Resuming from Peak Checkpoint: {LATEST}")
else:
  print("⚠️ Checkpoint 160 not found. Using default latest search.")

## Step 6: GRPO Training (Resuming up to Step 600, 7B Model)

In [ ]:
!python training/train_grpo.py \
    --model_id "Qwen/Qwen2.5-7B-Instruct" \
    --output_dir "{DRIVE_PATH}" \
    --num_episodes 1000 \
    --batch_size 8 \
    --term medium \
    --max_steps 600 {RESUME_ARG}

## Step 7: GPU Memory Reset
**Critical for 7B:** Run after training to clear VRAM.

In [ ]:
import torch, gc
if "trainer" in globals(): del trainer
gc.collect()
torch.cuda.empty_cache()
print("✨ GPU Memory Cleared.")

## Step 8: Multi-Checkpoint Evaluation (OOS 2023-24)

In [ ]:
import os, json
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Multi-source: [Restored Drive] 160-280, [HF Cloud] 300, [Current Drive] 460-600
CHECKPOINT_STEPS = [160, 180, 200, 220, 240, 260, 280, 300, 460, 480, 500, 520, 540, 560, 580, 600]
results_map = {}

print(f'Starting Multi-Checkpoint Evaluation for {len(CHECKPOINT_STEPS)} stages...')

for step in CHECKPOINT_STEPS:
    ckpt_path = os.path.join(DRIVE_PATH, f'checkpoint-{step}')
    is_remote = False
    
    # Fallback: Step 300 may be missing locally - pull from misnamed HF repo
    if step == 300 and not os.path.exists(ckpt_path):
        print('\n☁️  Fetching 300-step weights from Hugging Face Hub (bawsi99/indicators-grpo-qwen1.5b)...')
        ckpt_path = 'bawsi99/indicators-grpo-qwen1.5b'
        is_remote = True
    
    if not is_remote and not os.path.exists(ckpt_path):
        print(f'\n⚠️  Skipping step {step}: Checkpoint not found at {ckpt_path}')
        continue
    
    # Only run baseline on the very first checkpoint - skip for all subsequent ones
    skip_baseline = '--skip_baseline' if len(results_map) > 0 else ''
    source_label = '(Cloud Weights from HF)' if is_remote else ('(Skipping Baseline)' if skip_baseline else '(Full Eval)')
    print(f'\n--- Evaluating Checkpoint {step} {source_label} ---')
    
    !python evaluation/eval_finetuned.py \
        --adapter_path "{ckpt_path}" \
        --n_eval 50 \
        --term medium {skip_baseline}
    
    # Load results - for remote HF IDs, eval_finetuned.py resolves to local repo slug folder
    res_dir = 'indicators-grpo-qwen1.5b' if is_remote else ckpt_path
    metrics_file = os.path.join(res_dir, 'eval_results_full.json')
    
    if os.path.exists(metrics_file):
        with open(metrics_file, 'r') as f:
            results_map[step] = json.load(f)
            print(f'✅ Successfully loaded metrics for step {step}')
    else:
        print(f'❌ Failed to locate results at {metrics_file}')


## Step 9: Visualize Learning Progress

In [ ]:
import matplotlib.pyplot as plt
if "results_map" in globals() and results_map:
    steps = sorted(results_map.keys())
    accs = [results_map[s]["finetuned_grpo"]["overall_accuracy"] for s in steps]
    plt.figure(figsize=(10, 4))
    plt.plot(steps, accs, marker="o", label="Fine-tuned (GRPO)")
    b = results_map[steps[0]].get("baseline_zeroshot", {})
    if "overall_accuracy" in b: plt.axhline(y=b["overall_accuracy"], color="red", linestyle="--", label="Baseline")
    plt.title("Learning Curve"); plt.legend(); 
    plot_path = os.path.join(DRIVE_PATH, "grpo_learning_curve.png")
    plt.savefig(plot_path, dpi=150); plt.show()
    print(f"✅ Chart saved to: {plot_path}")
else: print("No results to plot.")

## Step 10: Consolidate All Results
Gather all individual JSONs from Drive into one master report.

In [ ]:
import json, os
master_results = {}
for folder in sorted(os.listdir(DRIVE_PATH)):
    if folder.startswith("checkpoint-"):
        f_path = os.path.join(DRIVE_PATH, folder, "eval_results_full.json")
        if os.path.exists(f_path):
            with open(f_path, "r") as f: master_results[folder.split("-")[1]] = json.load(f)
out_path = os.path.join(DRIVE_PATH, "consolidated_eval_report.json")
with open(out_path, "w") as f: json.dump(master_results, f, indent=2)
print(f"✅ Consolidated JSON saved to: {out_path}")